#### CUDA debugging commands


| Command | Purpose |
| :--- | :--- |
| `torch.cuda.is_available()` | Check CUDA availability |
| `torch.cuda.get_device_name(0)` | Identify GPU |
| `torch.cuda.get_device_properties(0)` | Get GPU specifications |
| `torch.cuda.memory_allocated()` | Check allocated memory |
| `torch.cuda.memory_reserved()` | Check reserved memory |
| `torch.cuda.max_memory_allocated()` | Check peak allocated memory |
| `torch.cuda.empty_cache()` | Release unused cached memory |
| `torch.cuda.synchronize()` | Wait for GPU operations |
| `torch.cuda.memory_summary()` | Inspect memory statistics |
| `torch.cuda.reset_peak_memory_stats()` | Reset peak memory statistics |
| `torch.cuda.is_current_stream_capturing()` | Check CUDA graph capture state |

#### Useful terminal debugging commands

- nvidia-smi
- nvidia-smi -l 1
- nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu,temperature.gpu --format=csv
- nvcc --version
- python -c "import torch; print(torch.__version__, torch.version.cuda)"
- lscpu | grep -i cache # check L2 Cache on Linux
- lscpu -C # Cache on Linux
- nvidia-smi -q | grep -i -A 5 "L2 Cache" # Cache on Linux NVIDIA

In [2]:
import torch

print("1. CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("2. Device Name:", torch.cuda.get_device_name(0))
    print("3. Device Properties:", torch.cuda.get_device_properties(0))
    print("4. Memory Allocated (bytes):", torch.cuda.memory_allocated())
    print("5. Memory Reserved (bytes):", torch.cuda.memory_reserved())
    print("6. Max Memory Allocated (bytes):", torch.cuda.max_memory_allocated())
    
    # Utility & state commands
    torch.cuda.empty_cache()
    print("7. Cache Emptied: Done")
    
    torch.cuda.synchronize()
    print("8. Synchronized: Done")
    
    print("9. Memory Summary:\n", torch.cuda.memory_summary())
    
    torch.cuda.reset_peak_memory_stats()
    print("10. Peak Memory Stats Reset: Done")
    
    print("11. Is Current Stream Capturing:", torch.cuda.is_current_stream_capturing())
else:
    print("No CUDA-enabled GPU detected on this system.")


1. CUDA Available: True
2. Device Name: NVIDIA GB10
3. Device Properties: _CudaDeviceProperties(name='NVIDIA GB10', major=12, minor=1, total_memory=124544MB, multi_processor_count=48, uuid=18e40491-52d1-86e9-74f2-935178843041, pci_bus_id=1, pci_device_id=0, pci_domain_id=15, L2_cache_size=24MB)
4. Memory Allocated (bytes): 0
5. Memory Reserved (bytes): 0
6. Max Memory Allocated (bytes): 400556032
7. Cache Emptied: Done
8. Synchronized: Done
9. Memory Summary:
 |===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|------------------------------------------------------------------

# CUDA Device Properties Explanation

## Device Information

```text
_CudaDeviceProperties(
    name='NVIDIA GB10',
    major=12,
    minor=1,
    total_memory=124544MB,
    multi_processor_count=48,
    uuid=18e40491-52d1-86e9-74f2-935178843041,
    pci_bus_id=1,
    pci_device_id=0,
    pci_domain_id=15,
    L2_cache_size=24MB
)
```

## 1. GPU Name

```text
name='NVIDIA GB10'
```

- GPU model detected by CUDA.
- Belongs to NVIDIA's latest **Blackwell architecture** family.

## multi_processor_count=48

Number of Streaming Multiprocessors (SMs). SM Count = 48
```text
GPU
 ├── SM0
 ├── SM1
 ├── SM2
 ├── ...
 └── SM47
```

## 2. Compute Capability

```text
major=12
minor=1
```

Compute Capability:

```text
12.1
```

The Compute Capability determines:

- Supported CUDA instructions
- Tensor Core features
- FP16/BF16/FP8 support
- Hardware optimizations

| GPU | Compute Capability |
|------|------|
| GTX 1080 | 6.1 |
| RTX 3090 | 8.6 |
| RTX 4090 | 8.9 |
| H100 | 9.0 |
| GB10 | 12.1 |

## 3. Total GPU Memory (VRAM)

```text
total_memory=124544MB
```
Conversion:

```text
124544 MB ÷ 1024 ≈ 121.6 GB
```

Available VRAM:

```text
≈ 122 GB
```

This allows:

- Large Language Model (LLM) inference
- Large batch training
- Massive vector databases
- Large-scale

##### For asynchronous CUDA error debugging:

CUDA_LAUNCH_BLOCKING=1 python train.py

#### Recommended preparation priority

##### Priority 1 – Must prepare Critical
- CUDA OOM and GPU memory management
- CPU vs GPU tensor device mismatch
- CUDA asynchronous execution
- CUDA_LAUNCH_BLOCKING=1
- Mixed precision and AMP
- Gradient accumulation and checkpointing
- GPU profiling and performance optimization
### Priority 2 – Strong technical knowledge *(Important)*
* CUDA streams and synchronization
* DataLoader optimization
* BF16 vs FP16
* GPU memory fragmentation
* Illegal memory access
* Low GPU utilization debugging
### Priority 3 – Advanced *(Advanced)*

* CUDA Graphs
* NCCL and DDP
* Tensor and pipeline parallelism
* CUDA race conditions
* Advanced GPU profiling

### 24 MB L2 Cache help ML:

### Memory Hierarchy Overview

| Memory Tier | Speed & Size | Purpose |
| :--- | :--- | :--- |
| **L1 Cache** | Smallest & Fastest | Very frequently accessed data |
| **L2 Cache — 24 MB** | Fast Memory | Frequently reused tensors and instructions |
| **L3 Cache** | Larger shared cache | Shared CPU cache (if available) |
| **RAM / System Memory** | Larger capacity, higher latency | Main system memory for host data and processes |
| **GPU VRAM** | High-bandwidth device memory | GPU memory for CUDA tensors and model parameters |

Conceptual hierarchy. CPU and GPU have separate cache and memory architectures. During these operations, processors repeatedly access:
- Model weights
- Input tensors
- Intermediate activations
- Gradients
- Optimizer states
If frequently accessed data fits in cache, the processor can reuse it without repeatedly fetching it from RAM or other memory levels.

### 3. Example: Matrix multiplication

Consider:

$$C = A \times B$$

Suppose:

* $A = 1024 \times 1024$
* $B = 1024 \times 1024$
* $\text{FP32} = 4\text{ bytes per element}$

Memory required:

$$1024 \times 1024 \times 4 = 4\text{ MB}$$

| Tensor | Memory |
| :--- | :--- |
| **Matrix A** | 4 MB |
| **Matrix B** | 4 MB |
| **Matrix C** | 4 MB |
| **Total** | **12 MB** |

Your 24 MB L2 cache could potentially accommodate these three matrices, assuming the cache is available to this workload and no significant competing cache usage.

However, actual matrix multiplication performance depends on cache blocking, hardware execution units, memory bandwidth, and parallelism.

#### Q. How to use L2 Cache in PyTorch?
You generally don't manually allocate L2 cache. The CPU/GPU hardware manages it automatically. You can optimize cache utilization through:
1. Batch processing: Choose appropriate batch sizes.
2. Memory locality: Access tensors sequentially where possible.
3. Tensor reuse: Avoid unnecessary memory allocations.
4. Contiguous tensors: Use .contiguous() when required by your operations.
5. Cache-friendly algorithms: Use tiled or blocked matrix multiplication.

#### Important: CPU L2 vs GPU L2 Cache
- NVIDIA GB10 with CUDA 13.0, one important distinction:
    - CPU L2 cache: Helps CPU-side data processing, preprocessing and CPU computations.
    - GPU L2 cache: Helps CUDA kernels reuse data and reduce repeated GPU memory accesses.
    - GPU VRAM: Stores model parameters, activations, gradients and other tensors.

For ML training, 24 MB L2 cache is a useful hardware resource, but it does not mean you can fit a 24 MB model or that training will be 24 MB faster. Its actual benefit depends on whether the workload reuses data that fits in the cache.

## PyTorch GPU & CUDA – Top 30 Interview Questions

**Recommended study list** · 30 questions · 7 topic areas

### Section 1. CUDA and GPU Fundamentals
* 
    1. What is CUDA, and how does it work with PyTorch?
        - `CUDA` (Compute Unified Device Architecture) is a `parallel computing platform` and programming model developed by NVIDIA that allows software to use `GPUs` for  `general-purpose processing`. In deep learning, `PyTorch uses CUDA to offload heavy mathematical operations—such as matrix multiplications and convolutions—from the CPU     to thousands of GPU cores simultaneously`.
    2. What is the difference between CPU and GPU architecture?
        - `CPU` (Central Processing Unit): Designed for general-purpose computing, complex logic, sequential execution and multitasking.
        - `GPU` (Graphics Processing Unit): Designed for massively parallel computing, executing thousands of similar operations simultaneously.
    3. What is the difference between CPU tensors and CUDA tensors?
        - A CPU tensor stores and performs computations using system RAM and the CPU processor, whereas a CUDA tensor stores data in GPU memory (VRAM) and performs computations    using the NVIDIA GPU.
    4. How do you check CUDA availability, GPU specifications, and memory in PyTorch?
    5. How do you move a model and tensors from CPU to GPU? What happens internally?

    In PyTorch, we use .to("cuda") or .cuda() to move models and tensors from CPU memory (RAM) to GPU memory (VRAM). When moved to GPU, the tensor data is `transferred from    system RAM to GPU VRAM`, and subsequent operations can execute on the GPU.

**Move a tensor from CPU to GPU**
```python
x_gpu = x.to("cuda")
```
**Move an entire model from CPU to GPU**
```python
# Create model on CPU
model = nn.Sequential(
    nn.Linear(100, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)
# Move model to GPU
device = torch.device("cuda")
model = model.to(device)
```
- CPU memory allocation: The original tensor is created in system RAM.
- GPU memory allocation: PyTorch allocates memory for the destination tensor in GPU VRAM.
- Data transfer: Tensor data is copied from CPU RAM to GPU VRAM, typically through PCIe.
- CUDA tensor creation: PyTorch returns a tensor whose data resides on the GPU.
- GPU computation: Subsequent supported operations execute on the GPU.
#### Q. Does .to("cuda") delete the original CPU tensor?

NO, Both tensors exist independently.

#### Q. What is a Streaming Multiprocessor (SM)?

### Streaming Multiprocessor (SM)

- 
    - A **Streaming Multiprocessor (SM)** is the fundamental processing unit inside an NVIDIA GPU. It executes thousands of parallel threads simultaneously to perform    high-throughput computations such as:

    * **Matrix multiplication**
    * **Tensor operations**
    * **Deep learning workloads**

    **Analogy:** Think of an SM as a *mini-parallel processor* inside the GPU.

    - GPU = Collection of multiple SMs.
    - SM = Parallel execution unit.
    - CUDA Core = Arithmetic processing unit inside an SM.
    - Tensor Core = Specialized unit for accelerated matrix operations.

## Readme:

When developing LLMs in PyTorch, `we generally don't manually assign computations to CUDA Cores or Tensor Cores`. PyTorch and CUDA select suitable GPU kernels automatically. However, we can influence hardware utilization through precision selection, mixed precision training, tensor dimensions and optimized libraries. FP16, BF16 and TF32 matrix operations can leverage Tensor Cores, while general-purpose operations typically execute on CUDA Cores.

## CUDA Cores Vs Tensor Cores
- **CUDA Cores vs Tensor Cores** — Both are processing units inside an NVIDIA GPU, but they are designed for different types of computations.
    * **CUDA Cores:** General-purpose parallel processing for arithmetic and other operations.
    * **Tensor Cores:** Specialized hardware designed to accelerate matrix multiplication and multiply-accumulate operations, particularly useful in Deep Learning.

#### - How to switch between CUDA and Tensor Core ?

LLMs using PyTorch, CUDA Cores and Tensor Cores are generally utilized automatically by the GPU runtime and PyTorch, but you can influence which hardware gets used through data types, operations, libraries, and configuration by:

- Automatic hardware selection: PyTorch, CUDA and cuBLAS select suitable GPU execution paths.
- Developer control: You influence hardware utilization by choosing FP32, FP16, BF16, TF32, operation types, and precision settings.

#### Auto control:
- 
    - if your code "output = model(input_ids)" then
    
        | LLM operation | Typical GPU execution |
        | :--- | :--- |
        | Linear layers | Tensor Cores when eligible |
        | Q × Kᵀ attention | Tensor Cores when eligible |
        | Attention × V | Tensor Cores when eligible |
        | Element-wise operations | CUDA Cores |
        | Activation functions | CUDA Cores / specialized units |
        | Normalization | CUDA Cores and optimized kernels |
        | Memory transfers | GPU memory subsystem |
    
    - The actual execution depends on data types, tensor dimensions, GPU architecture and selected kernels.

#### Use control

##### Option A: FP32 (32-bit precision):
FP32 operations generally use CUDA Cores, but NVIDIA GPUs can also accelerate FP32 matrix multiplication using `Tensor Cores` through TF32, when enabled and supported.
```python
model = model.cuda().float()
x = torch.randn(1024, 1024, device="cuda")
output = model(x)
```
##### Option B: FP16 (16-bit precision):
- This makes eligible matrix operations candidates for `Tensor Core` acceleration.
```python
model = model.cuda().half()
x = torch.randn(
    1024, 1024,
    device="cuda",
    dtype=torch.float16
)
output = model(x)
```
##### Option C: BF16 (Brain Floating Point 16):
- BF16 is particularly useful for LLM training because it provides a wider numerical range than FP16.
```python
model = model.cuda().to(torch.bfloat16)

x = torch.randn(
    1024, 1024,
    device="cuda",
    dtype=torch.bfloat16
)

output = model(x)
```
##### Option D: Automatic Mixed Precision
```python
import torch

model = model.cuda()

optimizer = torch.optim.AdamW(model.parameters())

for input_data, target in dataloader:

    input_data = input_data.cuda()
    target = target.cuda()

    optimizer.zero_grad()

    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16
    ):
        output = model(input_data)
        loss = loss_fn(output, target)

    loss.backward()
    optimizer.step()
```

## Explain GPU occupancy:

- Q. In pytorch can we control over SM?
Yes, but PyTorch does not provide a direct API to assign a CUDA operation to a specific Streaming Multiprocessor (SM). However, you can influence SM utilization through CUDA kernel configuration, workload design, and GPU optimization.
TODO

##### what is warp?
A **Warp** is a group of **32 threads** that execute the same instruction together on an NVIDIA GPU.

It is the fundamental scheduling unit inside a Streaming Multiprocessor (SM).

In simple terms:

- Thread = Executes an individual task.
- Warp = Group of 32 threads.
- SM = Executes and schedules multiple warps.

### 1. How does a Warp work?

Suppose you launch 128 CUDA threads.

CUDA divides them into warps:

$$\text{Number of Warps} = \frac{128}{32} = 4$$

#### What is Warp Divergence?
Warp divergence happens when threads within the same warp take different execution paths due to conditional statements.

### 1. Why do we need GPU occupancy?

**GPU Occupancy** is the ratio of active warps executing on a Streaming Multiprocessor (SM) to the maximum number of warps that the SM can support.

In simple terms, **GPU occupancy tells us how effectively an SM is utilizing its available thread execution capacity.** GPU occupancy measures how many warps are active on an SM relative to its hardware limit. It helps hide memory latency and improve hardware utilization, but maximum occupancy does not guarantee maximum performance.

### 1. Why do we need GPU occupancy?

An SM executes multiple warps simultaneously. Sometimes a warp must wait for data from GPU memory. To avoid keeping the SM idle, the GPU scheduler switches to another ready warp. Higher occupancy provides more opportunities to hide memory latency and keep GPU execution units busy.

### 2. How do we calculate GPU occupancy?

$$Occupancy = \frac{\text{Active Warps}}{\text{Maximum Warps}} \times 100$$

Example:

Suppose an SM supports a maximum of 64 resident warps.

| Active warps | Occupancy |
| :--- | :--- |
| 16 | 25% |
| 32 | 50% |
| 48 | 75% |
| 64 | 100% |

## PyTorch GPU Memory Management

PyTorch manages GPU memory through CUDA and its `Caching Allocator`. When you create a tensor on a CUDA device, PyTorch allocates GPU memory to store that tensor. You can allocate GPU memory explicitly by creating tensors on a CUDA device or by using CUDA memory APIs.

### Caching Allocator in CUDA Memory Management

The **CUDA Caching Allocator** is a memory management technique used by frameworks like **PyTorch** and **TensorFlow** to improve GPU memory allocation performance.

## Why is it needed?

GPU memory allocation (`cudaMalloc`) and deallocation (`cudaFree`) are expensive operations because:

- They require synchronization with the GPU.
- Frequent allocations/deallocations can slow down training.
- They can cause memory fragmentation.

Instead of returning memory to CUDA immediately, a caching allocator keeps freed memory blocks in a cache and reuses them later.

##### Traditional CUDA Allocation vs Caching Allocator Workflow
- Every allocation and deallocation goes through CUDA runtime.
```text
Request Tensor Memory
        |
        v
    cudaMalloc()
        |
        v
    Use Memory
        |
        v
     cudaFree()
```
##### Caching Allocator
```text
Request Tensor Memory
        |
        v
Is suitable cached block available?
      /    \
    Yes     No
     |       |
     v       v
 Reuse     cudaMalloc()
 Memory       |
     \         /
      \       /
       v     v
      Use Memory
           |
           v
     Tensor Deleted
           |
           v
 Store Block in Cache
 (not immediately freed)
```

#### Example PyTorch allocate GPU memory

In [3]:
import torch

# Check GPU availability
print(torch.cuda.is_available())

# Select GPU
device = torch.device("cuda:0")

# Allocate GPU memory
x = torch.randn(1024, 1024, device=device)

print(x.device)
print(x.shape)

True
cuda:0
torch.Size([1024, 1024])


#### In above Example What happens internally?

<div style="max-width: 480px; margin: auto; font-family: sans-serif; text-align: center;">

  <div style="background-color: #eef2ff; border: 1px solid #d1d5db; border-radius: 6px; padding: 14px;">
    <div style="font-weight: bold; font-size: 16px; color: #111827;">PyTorch Application</div>
    <div style="font-size: 13px; color: #4b5563; margin-top: 4px;">torch.randn()</div>
  </div>

  <div style="margin: 8px 0; color: #6b7280; font-size: 16px;">↓</div>

  <div style="background-color: #ffffff; border: 1px solid #d1d5db; border-radius: 6px; padding: 14px;">
    <div style="font-weight: bold; font-size: 16px; color: #111827;">CUDA Runtime</div>
    <div style="font-size: 13px; color: #4b5563; margin-top: 4px;">Requests GPU memory</div>
  </div>

  <div style="margin: 8px 0; color: #6b7280; font-size: 16px;">↓</div>

  <div style="background-color: #ffffff; border: 1px solid #d1d5db; border-radius: 6px; padding: 14px;">
    <div style="font-weight: bold; font-size: 16px; color: #111827;">PyTorch CUDA Caching Allocator</div>
    <div style="font-size: 13px; color: #4b5563; margin-top: 4px;">Finds a reusable block or obtains memory from CUDA</div>
  </div>

  <div style="margin: 8px 0; color: #6b7280; font-size: 16px;">↓</div>

  <div style="background-color: #e6f4ea; border: 1px solid #d1d5db; border-radius: 6px; padding: 14px;">
    <div style="font-weight: bold; font-size: 16px; color: #111827;">GPU VRAM</div>
    <div style="font-size: 13px; color: #4b5563; margin-top: 4px;">Tensor data is stored in GPU memory</div>
  </div>

</div>

``PyTorch generally reuses cached memory instead of requesting new GPU memory for every tensor allocation.``

#### where is cache memory is present ? It is hardware like L1,L2,L3 cache ?

No, the CUDA caching allocator cache is completely different from hardware caches like L1, L2, and CPU L3 cache. The CUDA caching allocator works at the Global Memory (VRAM) level, not at the L1/L2 cache level.
```text
GPU Memory Hierarchy

Registers          ← Fastest
   ↓
Shared Memory/L1 Cache
   ↓
L2 Cache
   ↓
Global Memory (VRAM)
```

#### Allocate GPU memory in PyTorch:

**Method 1: Allocate a tensor directly on GPU**

```python
x = torch.randn(1024, 1024, device="cuda")
```

**Method 2: Create on CPU and move to GPU**

```python
x = torch.randn(1024, 1024)

# Move tensor to GPU
x = x.to("cuda")
```

**Method 3: Move an existing model to GPU**

```python
import torch.nn as nn

model = nn.Linear(1024, 512)

model = model.to("cuda")
```

**Method 4: Allocate an empty GPU tensor**

```python
x = torch.empty(
    (1024, 1024),
    device="cuda",
    dtype=torch.float32
)
```

## Difference between allocated and reserved memory

These measure **different things**.

| Memory Type | Meaning |
| :--- | :--- |
| **Allocated Memory** | Memory currently occupied by tensors |
| **Reserved Memory** | Memory reserved by PyTorch's CUDA caching allocator from the GPU driver |

#### Relationship

```text
Reserved Memory >= Allocated Memory
cached_memory = reserved_memory - allocated_memory
```

PyTorch keeps extra GPU memory in a cache to avoid expensive GPU memory allocations/deallocations.

```text
GPU Memory Reserved by PyTorch
│
├── Allocated (used by tensors)
│
└── Cached/Unused (available for reuse)

This makes future allocations much faster.
```
- Before Tensor:

          ```text
          Reserved = 0 MB
          Allocated = 0 MB
          ```

- After Tensor Creation:

          ```text
          Reserved = 512 MB
          Allocated = 381 MB
          
          +------------------------+
          
          | Tensor Data 381 MB     |
          +------------------------+
          
          | Cache 131 MB           |
          +------------------------+
          ```

- After `del x`:

          ```text
          Reserved = 512 MB
          Allocated = 0 MB
          
          +------------------------+
          
          | Cache 512 MB           |
          +------------------------+
          ```

- After `empty_cache()`:

          ```text
          Reserved = 0 MB
          Allocated = 0 MB
          ```
- memory_allocated() → current tensor memory
- memory_reserved() → current cache + tensor memory
- max_memory_allocated() → peak tensor memory
- max_memory_reserved() → peak reserved memory

In [4]:
import torch

x = torch.randn(10000, 10000, device="cuda")

print(torch.cuda.memory_allocated() / 1024**2, "MB  actually storing tensor data")
print(torch.cuda.memory_reserved() / 1024**2, "MB  cached for future allocations")
print("After Deleting Tensor")
del x
print(torch.cuda.memory_allocated() / 1024**2, "MB  After Deleting Tensor actually storing tensor data")
print(torch.cuda.memory_reserved() / 1024**2, "MB  After Deleting Tensor cached for future allocations")

print('Releasing Cached Memory')
torch.cuda.empty_cache()
print(torch.cuda.memory_allocated() / 1024**2, "MB  Releasing Cached Memory actually storing tensor data")
print(torch.cuda.memory_reserved() / 1024**2, "MB  Releasing Cached Memory cached for future allocations")

print('(or much smaller) This returns unused cached blocks back to the CUDA driver.')

382.0 MB  actually storing tensor data
402.0 MB  cached for future allocations
After Deleting Tensor
0.0 MB  After Deleting Tensor actually storing tensor data
402.0 MB  After Deleting Tensor cached for future allocations
Releasing Cached Memory
0.0 MB  Releasing Cached Memory actually storing tensor data
0.0 MB  Releasing Cached Memory cached for future allocations
(or much smaller) This returns unused cached blocks back to the CUDA driver.


## `torch.cuda.empty_cache()` Does it release all GPU memory?

No. `torch.cuda.empty_cache()` does NOT release all GPU memory.

It only releases **unused cached memory** held by PyTorch's CUDA caching allocator.

### Example

```python
import torch

x = torch.randn(10000, 10000, device="cuda")

print("Allocated:", torch.cuda.memory_allocated()/1024**2)
print("Reserved :", torch.cuda.memory_reserved()/1024**2)

torch.cuda.empty_cache()

print("Allocated:", torch.cuda.memory_allocated()/1024**2)
print("Reserved :", torch.cuda.memory_reserved()/1024**2)
```

Output might be:

```text
Allocated: 381 MB
Reserved : 512 MB

Allocated: 381 MB
Reserved : 512 MB
```

Nothing changed because the tensor `x` still exists.

#### To actually free memory

```python
del x
torch.cuda.empty_cache()
```

Now:

```text
Allocated: 0 MB
Reserved : 0 MB
```

(or very close to 0)
### What `empty_cache()` frees

```text
Reserved Memory
├── Allocated Memory   ❌ Not freed
└── Cached Memory      ✅ Freed
```

Mathematically:

```text
Reserved = Allocated + Cached
```

After:

```python
torch.cuda.empty_cache()
```

```text
Reserved = Allocated
```

because cached blocks are returned to CUDA.



Priority 1 — Master thoroughly

RAG and LangChain

LoRA and QLoRA

Transformer architecture

Priority 2 — Strong technical understanding

- PyTorch fundamentals and Autograd
- GPU architecture and CUDA
- GPU memory management
- Mixed precision and optimization

Priority 3 — Practical understanding

- CUDA debugging and profiling
- Distributed training
- LLM deployment
- Agentic AI


### Section 2. GPU Memory Management
6. How does PyTorch manage GPU memory? Explain allocated vs reserved memory.
7. What is GPU Out-of-Memory (OOM)? What causes it, and how do you debug it?
8. What is `torch.cuda.empty_cache()`? Does it release all GPU memory?
9. What is gradient accumulation, and how does it reduce GPU memory consumption?
10. What is gradient checkpointing, and how does it trade computation for memory?

### Section 3. CUDA Errors and Debugging
11. How do you debug CUDA errors in PyTorch? What is `CUDA_LAUNCH_BLOCKING=1`?
12. What causes device mismatch errors between CPU and GPU tensors?
13. What causes `CUDA error: device-side assert triggered`?
14. What causes illegal memory access in CUDA, and how do you troubleshoot it?
15. How do you debug CUDA, NVIDIA driver, and PyTorch version compatibility issues?

### Section 4. GPU Performance Optimization
16. Why might GPU utilization remain low during model training?
17. How do you identify GPU performance bottlenecks using `torch.profiler`?
18. What is the difference between compute-bound and memory-bound operations?
19. What are `pin_memory=True`, `num_workers`, and `non_blocking=True` in DataLoader?
20. How do you measure GPU execution time accurately when CUDA operations are asynchronous?

### Section 5. Mixed Precision and Numerical Stability
21. What is mixed precision training, and how does AMP work in PyTorch?
22. What is the difference between FP32, FP16, BF16, and TF32?
23. What is `torch.autocast()` and `GradScaler`? Why are they used?
24. What causes gradient underflow, overflow, NaN, and Inf during GPU training?

### Section 6. CUDA Streams and Parallel Computing
25. What is CUDA synchronization, and why do we use `torch.cuda.synchronize()`?
26. What are CUDA streams, and how do they help overlap computation and data transfer?
27. What is the difference between Data Parallelism, Model Parallelism, and DistributedDataParallel (DDP)?

### Section 7. Practical GPU Debugging Scenarios
28. GPU memory keeps increasing after every training iteration. How do you identify and fix the memory leak?
29. Training works on CPU but fails or becomes slower on GPU. How do you investigate the problem?
30. GPU utilization is high, but training or inference is still slow. How do you identify and optimize the bottleneck?


#### Additional Questions

7. Important interview questions
For CPU/GPU architecture and ML optimization, prepare these questions:
1. What is the difference between CPU cores and GPU CUDA cores?
2. What is a Streaming Multiprocessor (SM)?
3. What is the difference between CUDA Cores and Tensor Cores?
4. What is GPU occupancy?
5. What is warp execution in CUDA?
6. What is memory coalescing?
7. What is the difference between shared memory and global memory?
8. What is L1, L2 and L3 cache?
9. What causes CPU-GPU memory transfer overhead?
10. Why is GPU not always faster than CPU for ML workloads?

## 5. PyTorch GPU, CUDA & Debugging – Interview Questions [100 questions]

**High-priority topics** · **Practical debugging**

---

### Section 1. CUDA and GPU Fundamentals
1. What is CUDA, and how does it work with PyTorch?
2. What is the difference between CPU and GPU architecture?
3. What is the difference between CPU tensors and CUDA tensors?
4. How do you check CUDA availability in PyTorch?
5. How do you check GPU name, memory, and CUDA version using PyTorch?
6. How do you move a model from CPU to GPU?
7. How do you move tensors between CPU and GPU?
8. What happens internally when you execute `tensor.to("cuda")`?
9. What is the difference between CUDA cores and Tensor Cores?
10. What is a CUDA kernel?
11. What is CUDA execution and why is it asynchronous?
12. What is the difference between CUDA Runtime and CUDA Driver?

---

### Section 2. GPU Memory Management and Optimization
*(Very important)*

13. How does PyTorch manage GPU memory?
14. What is the difference between allocated memory and reserved memory?
15. What is `torch.cuda.empty_cache()` and when should you use it?
16. What is GPU Out-of-Memory (OOM), and how do you debug it?
17. How do you reduce GPU memory consumption during training?
18. What is gradient accumulation, and how does it reduce memory requirements?
19. What is gradient checkpointing, and how does it save GPU memory?
20. What is the difference between `torch.no_grad()` and `torch.inference_mode()`?
21. How does `detach()` help prevent unnecessary computation graph memory usage?
22. What causes GPU memory leaks in PyTorch?
23. How do you identify which tensors are consuming GPU memory?
24. What is memory fragmentation in CUDA?
25. What is the difference between `torch.cuda.memory_allocated()` and `torch.cuda.memory_reserved()`?
26. How do you investigate memory consumption using `torch.cuda.memory_summary()`?

---

### Section 3. CUDA Errors and GPU Debugging
*(Highest practical importance)*

27. How do you debug CUDA errors in PyTorch?
28. What is `CUDA_LAUNCH_BLOCKING=1`, and why is it useful?
29. What causes CUDA out of memory errors, and how do you fix them?
30. What causes `RuntimeError: Expected all tensors to be on the same device`?
31. How do you fix device mismatch errors between CPU and CUDA tensors?
32. What causes `CUDA error: device-side assert triggered`?
33. What causes `CUDA error: an illegal memory access was encountered`?
34. How do you debug `CUBLAS_STATUS_ALLOC_FAILED`?
35. What causes `CUDA error: initialization error`?
36. What causes `RuntimeError: CUDA error: no kernel image is available for execution on the device`?
37. How do you debug PyTorch errors caused by incompatible CUDA, NVIDIA drivers, and PyTorch versions?
38. What causes `NaN` or `Inf` values during GPU training?
39. How do you debug training that works on CPU but fails on GPU?
40. How do you debug a CUDA error when the stack trace points to an unrelated operation?
41. How do you identify invalid tensor shapes, indices, and memory access during GPU execution?

---

### Section 4. GPU Performance Optimization
*(Frequently asked)*

42. Why might GPU utilization remain low during model training?
43. How do you identify GPU performance bottlenecks?
44. What is the difference between compute-bound and memory-bound operations?
45. What is GPU memory bandwidth?
46. What is the difference between GPU utilization and GPU memory utilization?
47. How do you profile GPU performance using `torch.profiler`?
48. How do you measure GPU execution time correctly?
49. Why is `torch.cuda.synchronize()` required for accurate CUDA timing?
50. How do you optimize CPU-to-GPU data transfers?
51. What is `pin_memory=True` in PyTorch DataLoader?
52. What is `non_blocking=True`, and how does it improve data transfer?
53. How do you optimize DataLoader performance for GPU training?
54. What is the difference between `num_workers=0` and `num_workers>0`?
55. What is CUDA Graphs, and how does it improve performance?
56. How do you identify CPU bottlenecks affecting GPU utilization?

---

### Section 5. Mixed Precision and Numerical Stability
57. What is mixed precision training?
58. What is Automatic Mixed Precision (AMP)?
59. What is the difference between FP32, FP16, BF16, and TF32?
60. What is the difference between `torch.float16` and `torch.bfloat16`?
61. What is `torch.autocast()`?
62. What is `GradScaler`, and why is it used?
63. What is gradient underflow and overflow?
64. How does mixed precision reduce GPU memory consumption?
65. What happens if you use FP16 without gradient scaling?
66. How do you debug `NaN` and `Inf` values in mixed precision training?
67. When should you prefer BF16 over FP16?

---

### Section 6. CUDA Streams, Synchronization and Parallel Execution
68. What is a CUDA stream?
69. What is the default CUDA stream?
70. What is the difference between synchronous and asynchronous CUDA operations?
71. How does `torch.cuda.synchronize()` work?
72. What happens when multiple CUDA streams execute simultaneously?
73. How do you use CUDA streams for overlapping computation and data transfer?
74. What is the difference between CUDA events and CUDA synchronization?
75. What causes race conditions in CUDA programming?
76. What is the role of CUDA events in GPU profiling?

---

### Section 7. Advanced GPU Architecture and Parallel Training
77. What is the difference between data parallelism and model parallelism?
78. What is DistributedDataParallel (DDP)?
79. What is the difference between DataParallel and DistributedDataParallel?
80. What is gradient synchronization in distributed GPU training?
81. What is NCCL, and how does PyTorch use it?
82. What is the difference between single-GPU and multi-GPU training?
83. What is GPU communication overhead?
84. What is tensor parallelism?
85. What is pipeline parallelism?
86. How do you debug multi-GPU training failures?
